# Sales Dataset: 10 Graph Questions

Upload this notebook to Google Colab, then run the cells from top to bottom. When prompted, select all eight CSV files: `customers.csv`, `employees.csv`, `order_items.csv`, `orders.csv`, `products.csv`, `returns.csv`, `stores.csv`, and `targets.csv`.

**Discount assumption:** the setup assumes discounts are fractions (for example, `0.10` means 10%). If your CSV stores `10` for 10%, change the indicated line in the setup cell.

## My question-selection approach

I reviewed the available CSV columns, looked for comparisons that were not in the original assignment, and matched each question to a graph that makes the comparison easy to understand. These questions explore patterns across customer, product, order, supplier, return, and employee data.

| # | Question | Why ask it? | Suggested graph |
|---|---|---|---|
| 1 | How does revenue vary by day of the week? | Helps identify busy and quiet shopping days for planning promotions and staffing. | Bar chart |
| 2 | Do customer age groups prefer different product categories? | Shows whether category demand differs across age groups. | Stacked bar chart |
| 3 | Which payment methods are used most, and which generate the most revenue? | Compares payment frequency with the sales value associated with each method. | Side-by-side bar charts |
| 4 | Are larger discounts associated with more units sold? | Explores whether discount levels and item quantities move together. | Scatter plot |
| 5 | Which suppliers contribute the most revenue and profit? | Helps compare suppliers by sales and profitability. | Grouped bar chart |
| 6 | How many days pass between a product launch and its first sale? | Shows how quickly newly launched products begin selling. | Scatter plot |
| 7 | Do long-time customers spend more than newer customers? | Compares signup timing with customer spending. | Scatter plot |
| 8 | Which customer cities have the most orders and highest average order value? | Separates order volume from typical order value by city. | Two bar charts |
| 9 | Are return reasons more common for particular product categories? | Helps spot return patterns that may be concentrated in certain categories. | Stacked bar chart |
| 10 | Is employee salary related to revenue generated? | Explores whether employee salary and sales revenue show a relationship. | Scatter plot |

The charts show patterns in this dataset; relationships in the charts do not by themselves prove that one factor caused another.


## Setup: Upload files and prepare data

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Read uploaded CSV files
customers = pd.read_csv("customers.csv")
employees = pd.read_csv("employees.csv")
order_items = pd.read_csv("order_items.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
returns = pd.read_csv("returns.csv")
stores = pd.read_csv("stores.csv")
targets = pd.read_csv("targets.csv")

# Join tables for sales analysis
sales = (
    order_items
    .merge(orders, on="OrderID", how="left")
    .merge(products, on="ProductID", how="left")
    .merge(customers, on="CustomerID", how="left")
    .merge(employees, on="EmployeeID", how="left", suffixes=("_Customer", "_Employee"))
)

# Parse date fields
sales["OrderDate"] = pd.to_datetime(sales["OrderDate"], errors="coerce")
customers["SignupDate"] = pd.to_datetime(customers["SignupDate"], errors="coerce")
products["LaunchDate"] = pd.to_datetime(products["LaunchDate"], errors="coerce")

# Assumes Discount uses fractions: 0.10 = 10%.
# If your file uses 10 = 10%, uncomment this line:
# order_items["Discount"] = order_items["Discount"] / 100
# Rebuild sales after that conversion if you enable it.
sales["Discount"] = pd.to_numeric(sales["Discount"], errors="coerce").fillna(0)
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce").fillna(0)
sales["Price"] = pd.to_numeric(sales["Price"], errors="coerce").fillna(0)
sales["Cost"] = pd.to_numeric(sales["Cost"], errors="coerce").fillna(0)
sales["Revenue"] = sales["Quantity"] * sales["Price"] * (1 - sales["Discount"])
sales["Profit"] = sales["Quantity"] * (sales["Price"] * (1 - sales["Discount"]) - sales["Cost"])

print(f"Sales data prepared: {len(sales):,} item rows")
display(sales.head())

## 1. How does revenue vary by day of the week?

In [ ]:
sales["DayOfWeek"] = sales["OrderDate"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_revenue = sales.groupby("DayOfWeek")["Revenue"].sum().reindex(weekday_order)

weekday_revenue.plot(kind="bar", color="steelblue", figsize=(9, 5))
plt.title("Revenue by Day of the Week")
plt.xlabel("Day")
plt.ylabel("Revenue")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 2. Do customer age groups prefer different product categories?

In [ ]:
sales["AgeGroup"] = pd.cut(sales["Age"], bins=[0, 19, 29, 39, 49, 59, 120], labels=["Under 20", "20–29", "30–39", "40–49", "50–59", "60+"])
age_category = sales.pivot_table(index="AgeGroup", columns="Category", values="Quantity", aggfunc="sum", fill_value=0, observed=False)
age_category.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Units Sold by Customer Age Group and Product Category")
plt.xlabel("Age group")
plt.ylabel("Quantity sold")
plt.xticks(rotation=0)
plt.legend(title="Category", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 3. Which payment methods are used most, and which generate the most revenue?

In [ ]:
order_counts = orders.groupby("PaymentMethod")["OrderID"].nunique().rename("OrderCount")
payment_revenue = sales.groupby("PaymentMethod")["Revenue"].sum().rename("Revenue")
payment_summary = pd.concat([order_counts, payment_revenue], axis=1).sort_values("Revenue", ascending=False)
display(payment_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
payment_summary["OrderCount"].plot(kind="bar", ax=axes[0], color="darkorange", title="Orders by Payment Method")
payment_summary["Revenue"].plot(kind="bar", ax=axes[1], color="seagreen", title="Revenue by Payment Method")
for ax in axes:
    ax.set_xlabel("Payment method")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 4. Are larger discounts associated with more units sold?

In [ ]:
discount_sales = sales.copy()
discount_sales["DiscountPercent"] = discount_sales["Discount"] * 100
plt.figure(figsize=(9, 5))
sns.scatterplot(data=discount_sales, x="DiscountPercent", y="Quantity", alpha=0.6)
plt.title("Discount Level and Quantity Sold")
plt.xlabel("Discount (%)")
plt.ylabel("Quantity on item row")
plt.tight_layout()
plt.show()
print("Correlation between discount and quantity:", discount_sales[["DiscountPercent", "Quantity"]].corr().iloc[0, 1])

## 5. Which suppliers contribute the most revenue and profit?

In [ ]:
supplier_summary = sales.groupby("Supplier")[["Revenue", "Profit"]].sum().sort_values("Revenue", ascending=False)
display(supplier_summary)
supplier_summary.plot(kind="bar", figsize=(12, 6), color=["royalblue", "orange"])
plt.title("Revenue and Profit by Supplier")
plt.xlabel("Supplier")
plt.ylabel("Amount")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 6. How many days pass between product launch and first sale?

In [ ]:
first_sales = sales.groupby("ProductID")["OrderDate"].min().rename("FirstSaleDate")
product_timing = products.merge(first_sales, on="ProductID", how="inner").dropna(subset=["LaunchDate", "FirstSaleDate"]).copy()
product_timing["DaysToFirstSale"] = (product_timing["FirstSaleDate"] - product_timing["LaunchDate"]).dt.days
product_timing = product_timing[product_timing["DaysToFirstSale"] >= 0]

plt.figure(figsize=(10, 5))
sns.scatterplot(data=product_timing, x="LaunchDate", y="DaysToFirstSale", hue="Category")
plt.title("Days from Product Launch to First Sale")
plt.xlabel("Launch date")
plt.ylabel("Days to first sale")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()
display(product_timing[["ProductName", "Category", "LaunchDate", "FirstSaleDate", "DaysToFirstSale"]].sort_values("DaysToFirstSale").head(10))

## 7. Do long-time customers spend more than newer customers?

In [ ]:
customer_spending = sales.groupby("CustomerID")["Revenue"].sum().rename("TotalRevenue").reset_index()
customer_spending = customer_spending.merge(customers[["CustomerID", "Name", "SignupDate"]], on="CustomerID", how="left").dropna(subset=["SignupDate"])
plt.figure(figsize=(10, 5))
sns.scatterplot(data=customer_spending, x="SignupDate", y="TotalRevenue", alpha=0.7)
plt.title("Customer Signup Date and Total Spending")
plt.xlabel("Signup date")
plt.ylabel("Total revenue")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 8. Which customer cities have the most orders and highest average order value?

In [ ]:
order_revenue = sales.groupby("OrderID")["Revenue"].sum().rename("OrderRevenue")
order_details = orders.merge(order_revenue, on="OrderID", how="left").merge(customers[["CustomerID", "City"]], on="CustomerID", how="left")
city_summary = order_details.groupby("City").agg(OrderCount=("OrderID", "nunique"), AverageOrderValue=("OrderRevenue", "mean")).sort_values("OrderCount", ascending=False)
display(city_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
city_summary["OrderCount"].sort_values(ascending=False).plot(kind="bar", ax=axes[0], color="teal", title="Orders by City")
city_summary["AverageOrderValue"].sort_values(ascending=False).plot(kind="bar", ax=axes[1], color="purple", title="Average Order Value by City")
for ax in axes:
    ax.set_xlabel("City")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 9. Are return reasons more common for particular product categories?

In [ ]:
returned_orders = returns[["OrderID", "Reason"]].drop_duplicates()
returned_items = order_items.merge(returned_orders, on="OrderID", how="inner").merge(products[["ProductID", "Category"]], on="ProductID", how="left")
reason_category = pd.crosstab(returned_items["Category"], returned_items["Reason"])
display(reason_category)
reason_category.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Returned Items by Product Category and Reason")
plt.xlabel("Product category")
plt.ylabel("Returned item rows")
plt.xticks(rotation=40, ha="right")
plt.legend(title="Reason", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 10. Is employee salary related to revenue generated?

In [ ]:
employee_revenue = sales.groupby("EmployeeID")["Revenue"].sum().rename("TotalRevenue").reset_index()
employee_revenue = employee_revenue.merge(employees[["EmployeeID", "Name", "Salary", "Region"]], on="EmployeeID", how="left")
plt.figure(figsize=(9, 5))
sns.scatterplot(data=employee_revenue, x="Salary", y="TotalRevenue", hue="Region", s=100)
plt.title("Employee Salary and Revenue Generated")
plt.xlabel("Salary")
plt.ylabel("Total revenue")
plt.tight_layout()
plt.show()
display(employee_revenue.sort_values("TotalRevenue", ascending=False).head(10))